In [8]:
import scipy.optimize as opt
import numpy as np
import obs_to_obs_seq_in as obsin
import moored_obs_generator as obsgen
import datetime as dt
import pydartdiags.obs_sequence.obs_sequence as obsq
import xarray as xr
import true_w_mean as truew
import make_uv_samples as makeuv
from pathlib import Path
import sys
import subprocess
import os
module_dir = Path("/glade/work/iranjan/tpose24-osse/")
sys.path.append(str(module_dir))

import osse_tools as ost

In [9]:
casename = 'EEP_MITgcm185Lvgrid_Whitt2026hgrid'
path = '/glade/derecho/scratch/iranjan/archive/' + casename + '/ocn/hist/'
files = casename + '.mom6.h.z.2015-0*-*.nc'

In [11]:
true_ds = xr.open_mfdataset(path+files,drop_variables=["average_DT",])
true_ds['uh'] = true_ds['umo']/1035.
true_ds['vh'] = true_ds['vmo']/1035. 
true_ds = true_ds.rename({'z_l':'zl','z_i':'zi'})
true_ds['w_est'] = true_ds['vert_remap_h_tendency'].cumsum(dim='zl')
weekly_ds = truew.weekly_mean_w_est_profile(true_ds, center_lat=0.5, center_lon=220.0, box_size=1)


In [ ]:


# Path to the bash runner (run_pmo_parallel.sh) — adjust if you place it
# elsewhere. Kept as a separate script rather than inline Python subprocess
# calls because it avoids per-worker Python process overhead, which matters
# when each PMO call is only ~6s.
PMO_SCRIPT = "/glade/work/iranjan/fast-osse/run_pmo_qint.sh"


def run_pmo_parallel(split_dir, max_concurrent=32):
    """
    Run PMO across all time-split subdirectories under split_dir, bounded
    to max_concurrent simultaneous processes. max_concurrent should match
    however many CPUs your qinteractive session actually holds — 32 is the
    qinteractive default; raise it only if you explicitly requested more
    (e.g. -l select=1:ncpus=128:mpiprocs=128).

    Raises RuntimeError if any individual PMO call failed, so a partial
    failure can't silently flow into the join/reshape step below.
    """
    result = subprocess.run(
        ["bash", PMO_SCRIPT, split_dir, str(max_concurrent)],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f"PMO failed in {result.returncode} subdirectorie(s) under "
            f"{split_dir}. See {split_dir}/pmo_logs/*.log for details.\n"
            f"stderr: {result.stderr}"
        )


def opt_loc(loc_list, true_w):
    obs_seq = obsin.create_obs_seq_in()

    # TO-DO: add as inputs?
    start_year, start_month, start_date, start_time = 2015, 1, 1, 1
    end_year, end_month, end_date, end_time = 2015, 1, 3, 23

    for lat, lon in loc_list:
        obsgen.MooredObs(
            obs_seq, lon, lat, 8, 80, 2, "height (m)",
            dt.datetime(start_year, start_month, start_date, start_time),
            dt.datetime(end_year, end_month, end_date, end_time),
            0.001, dt.timedelta(hours=3),
        )

    out_path = "/glade/derecho/scratch/iranjan/eep-osse-30"
    obsin.split_obs_seq_by_time(
        obs_seq, out_path, "EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z."
    )

    # --- run PMO across all time-split subdirectories in parallel ---
    # NOTE: confirm split_obs_seq_by_time writes chunk subdirectories
    # directly into out_path (not into out_path + "/outputs", which is a
    # separate directory join_obs_seq_outputs reads from below) — if the
    # bash script's `find` picks up the wrong directory, it'll just report
    # 0 subdirectories found rather than error, so check that printout.
    run_pmo_parallel(out_path, max_concurrent=32)

    joined_obs_seq = makeuv.join_obs_seq_outputs(out_path + "/outputs")
    uv_samples = makeuv.reshape_obs_to_uv_samples(joined_obs_seq.df)

    est_w = ost.compute_w_planefit(uv_samples, extrapolate_to_surface=False)

    # --- FIX: previously this collapsed to true_w + random noise, so the
    # optimizer was chasing noise instead of the real estimate. Use the
    # actual planefit output, averaged over the 3-hourly windows to match
    # true_w's shape. Confirm true_w's depth grid matches est_w['w_est']'s
    # depth coordinate before trusting this norm — if they're on different
    # grids you need the interpolation step from reshape_obs_to_uv_samples's
    # sibling optimize_glider_locations function.
    w_est = est_w['w_est'].mean('time').values

    return np.linalg.norm(true_w - w_est)

In [ ]:
loc_list = [(0, 219), (1, 219), (0, 221), (1, 221)]

In [7]:
result = opt_loc(loc_list, weekly_ds)

KeyboardInterrupt: 

In [ ]:
print(result)

In [ ]:
scipy_result = opt.minimize()